# Ch 01 · What Is MuJoCo Warp?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/01-what-is-mujoco-warp.ipynb)

Companion notebook for the chapter page `chapters/01-what-is-mujoco-warp.html` ([page](../chapters/01-what-is-mujoco-warp.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

## How to use this notebook
Run the two cells above first: `!nvidia-smi` shows whether the runtime has a GPU, and the setup cell checks the driver **before** installing, picks the right Warp build (PyPI `warp-lang==1.19.0` for an R580+ driver, otherwise NVIDIA's `+cu12` wheel), installs pinned versions with the `[viewer]` extra, sets `wp.config.log_level = wp.LOG_WARNING` right after `import warp`, and confirms that Warp itself sees the GPU (`ON_GPU`).

If it prints `WARNING: Warp sees no GPU`, use Runtime > Change runtime type > GPU and run it again. Every example also runs on CPU with small budgets (that is how the course is verified).

Pitfalls: the distribution is `mujoco-core` but the import name is `mujoco`; never also `pip install mujoco` on top.

## 1. Check the environment (chapter section 6)
Versions, the default Warp device, and the GPU count. Note that `md.version('mujoco')` fails: the distribution is called `mujoco-core`.

In [ ]:
# Example 1.1 - ex1_env_check.py
"""Example 1.1 - check the environment: versions, the Warp device and the GPU count.

Prints what is installed and which device Warp will use ("cpu" or "cuda:0").
Run: python ex1_env_check.py
"""
import importlib.metadata as md

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

wp.init()  # explicit init: finds the CUDA driver (or not) and picks the default device

print("mujoco.__version__        :", mujoco.__version__)       # import name: mujoco
print("dist mujoco-core          :", md.version("mujoco-core"))  # distribution name: mujoco-core
print("warp.__version__          :", wp.__version__)
print("dist mujoco-warp          :", md.version("mujoco-warp"))
print("numpy                     :", np.__version__)
print("default Warp device       :", wp.get_device())
print("CUDA devices Warp can see :", wp.get_cuda_device_count())
print("mjw.step is callable      :", callable(mjw.step))

# The distribution is called mujoco-core, so asking for "mujoco" fails (the import name is still mujoco).
try:
    md.version("mujoco")
except md.PackageNotFoundError as e:
    print("md.version('mujoco')      : PackageNotFoundError:", e)

if wp.get_cuda_device_count() == 0:
    print("No GPU: everything still runs on Warp's CPU device, just slowly (fine for this course's checks).")

## 2. The smallest model
The official tutorial's install check: an empty `<mujoco/>` moved to the device with `put_model`. On the host `opt.timestep` is a float64 number; on the device it is a float32 Warp array with a leading batch dimension.

In [ ]:
# Example 1.2 - ex2_empty_model.py
"""Example 1.2 - the smallest possible MJWarp model: an empty <mujoco/> moved to the device.

This is the tutorial's install check: if put_model works, MuJoCo, Warp and MJWarp all agree.
Run: python ex2_empty_model.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

mjm = mujoco.MjModel.from_xml_string("<mujoco/>")  # host model (C MuJoCo, float64)
m = mjw.put_model(mjm)                              # device model (Warp arrays, float32)

print("type(mjm)         :", type(mjm).__name__)
print("type(m)           :", type(m).__module__ + "." + type(m).__name__)
print("m.nq, m.nv        :", m.nq, m.nv)
print("m.nbody           :", m.nbody, "(the world body always exists)")
print("m.opt.timestep    :", type(m.opt.timestep).__name__, m.opt.timestep.dtype.__name__,
      m.opt.timestep.shape, m.opt.timestep.numpy())
print("mjm.opt.timestep  :", type(mjm.opt.timestep).__name__, mjm.opt.timestep)
print("device            :", m.opt.timestep.device)

## 3. BALL_DROP in classic C MuJoCo
One world, float64, on the CPU. The ball lands at about t = 0.43 s and rests at z = 0.0996 (contacts are soft, a tiny penetration is normal).

In [ ]:
# Example 1.3 - ex3_ball_c.py
"""Example 1.3 - BALL_DROP in classic C MuJoCo: one world, float64, on the CPU.

Drops a 0.1 m ball from 1 m for 1 s with mujoco.mj_step and prints its height every 0.1 s.
Run: python ex3_ball_c.py
"""
import mujoco
import numpy as np

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)  # MJCF -> compiled model (constants)
mjd = mujoco.MjData(mjm)                         # state of ONE world
print(f"nq={mjm.nq} nv={mjm.nv} ngeom={mjm.ngeom} timestep={mjm.opt.timestep}")
print("qpos0 =", mjm.qpos0, "(x y z, then the quaternion w x y z)")

nstep = round(1.0 / mjm.opt.timestep)            # 500 steps = 1 s
for i in range(1, nstep + 1):
    mujoco.mj_step(mjm, mjd)
    if i % 50 == 0:                              # every 0.1 s
        print(f"t={mjd.time:.1f} s  z={mjd.qpos[2]:.5f}  ncon={mjd.ncon}")

## 4. The same ball in MJWarp (nworld = 1)
`put_model` / `make_data(nworld=1)` / `mjw.step`, run next to C MuJoCo. The difference stays around 1e-6 m: float32 vs float64.

In [ ]:
# Example 1.4 - ex4_ball_mjw.py
"""Example 1.4 - the same BALL_DROP in MJWarp with nworld=1, compared with C MuJoCo.

put_model / make_data / mjw.step instead of MjData / mj_step. Prints the ball height every 0.1 s
and the largest difference from the C run (float32 vs float64).
Run: python ex4_ball_mjw.py [--dump-json PATH]
  --dump-json PATH writes both trajectories (every 0.01 s) for the page's comparison chart.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()  # parse_known_args: also works inside Jupyter

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
mjd = mujoco.MjData(mjm)                                    # C MuJoCo reference (float64)
m = mjw.put_model(mjm)                                      # copy constants to the device
d = mjw.make_data(mjm, nworld=1, nconmax=4, njmax=16)       # state for ONE world on the device

nstep = round(1.0 / mjm.opt.timestep)
z_c, z_w = [], []
for i in range(1, nstep + 1):
    mujoco.mj_step(mjm, mjd)
    mjw.step(m, d)
    if i % 5 == 0:                       # every 0.01 s; .numpy() copies to the host (fine here)
        z_c.append(float(mjd.qpos[2]))
        z_w.append(float(d.qpos.numpy()[0, 2]))
    if i % 50 == 0:
        print(f"t={i * mjm.opt.timestep:.1f} s  z_mjw={z_w[-1]:.5f}  z_c={z_c[-1]:.5f}")

diff = np.abs(np.array(z_c) - np.array(z_w))
print("d.qpos:", d.qpos.shape, d.qpos.dtype.__name__, "on", d.qpos.device)
print(f"max |z_mjw - z_c| over 1 s: {diff.max():.2e} m")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"dt": 0.01, "z_c": [round(v, 5) for v in z_c], "z_mjw": [round(v, 5) for v in z_w],
                   "diff": [float(f"{v:.3e}") for v in diff]}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(z_c), "samples")

## 5. Many worlds
1024 worlds on a GPU (8 on CPU), each with its own start height written in place with `d.qpos.assign`. One `mjw.step` advances them all.

In [ ]:
# Example 1.5 - ex5_ball_1024.py
"""Example 1.5 - many BALL_DROP worlds in one call: 1024 on a GPU, 8 on the CPU.

Each world starts at its own random height (written with d.qpos.assign); one mjw.step advances all.
Run: python ex5_ball_1024.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 8
NSTEP = 250                                   # 0.5 s: the highest balls are still falling

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)   # nconmax is per world

rng = np.random.default_rng(0)
qpos = np.tile(mjm.qpos0, (NWORLD, 1)).astype(np.float32)   # (nworld, nq)
qpos[:, 2] = rng.uniform(0.5, 2.0, NWORLD)                  # column 2 = ball z
d.qpos.assign(qpos)                                          # write in place on the device
print("device:", wp.get_device(), "| nworld:", d.nworld)
print("d.qpos.shape:", d.qpos.shape, "| first start heights:", qpos[:4, 2])

for _ in range(NSTEP):
    mjw.step(m, d)                            # ONE call advances every world
wp.synchronize()

z = d.qpos.numpy()[:, 2]
t = NSTEP * mjm.opt.timestep
print(f"t = {t:.2f} s  final z: min {z.min():.4f}  max {z.max():.4f}")
print("worlds resting on the floor (z < 0.101):", int((z < 0.101).sum()), "of", NWORLD)
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

## 5b. 256 double pendulums (the chapter's hero replay)
Start angles spread over 0.1 rad; chaos spreads them much further within 3 s. Run with `--dump-json` from a terminal to write the replay frames.

In [ ]:
# Example 1.5b - ex5b_pendulum_grid.py
"""Example 1.5b - 256 double pendulums, one launch per stage: the data behind the chapter's hero replay.

Every world starts horizontal with a slightly different shoulder angle (a spread of 0.1 rad).
The worlds never talk to each other, so they are free to drift apart (a double pendulum is chaotic).
Run: python ex5b_pendulum_grid.py [--dump-json PATH]
  --dump-json PATH writes 60 decimated frames of both hinge angles for all 256 worlds (Rule D).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

PENDULUM = """
<mujoco model="double_pendulum">
  <option timestep="0.01"><flag energy="enable"/></option>
  <worldbody>
    <light pos="0 -2 3" dir="0 1 -1"/>
    <camera name="side" pos="0 -3 1" xyaxes="1 0 0 0 0 1"/>
    <body name="link1" pos="0 0 1.5">
      <joint name="hinge1" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
      <body name="link2" pos="0 0 -.5">
        <joint name="hinge2" type="hinge" axis="0 1 0"/>
        <geom type="capsule" fromto="0 0 0 0 0 -.5" size=".04" mass="1"/>
        <site name="tip" pos="0 0 -.5"/>
      </body>
    </body>
  </worldbody>
  <actuator><motor name="shoulder" joint="hinge1" gear="1" ctrlrange="-5 5"/></actuator>
  <sensor>
    <jointpos name="q1" joint="hinge1"/>
    <jointvel name="v1" joint="hinge1"/>
    <framepos name="tip_pos" objtype="site" objname="tip"/>
  </sensor>
  <keyframe><key name="horizontal" qpos="1.5708 0"/></keyframe>
</mujoco>
"""

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

NWORLD = 256          # same on CPU and GPU: the replay needs exactly 16 x 16 worlds
NSTEP = 300           # 3 s at timestep 0.01
EVERY = 5             # keep every 5th step -> 60 frames

mjm = mujoco.MjModel.from_xml_string(PENDULUM)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)          # no contacts in this model: default sizes are fine

qpos = np.zeros((NWORLD, mjm.nq), dtype=np.float32)
qpos[:, 0] = np.pi / 2 + np.linspace(0.0, 0.1, NWORLD)   # shoulder: horizontal + 0..0.1 rad
d.qpos.assign(qpos)

frames = []
for i in range(1, NSTEP + 1):
    mjw.step(m, d)
    if i % EVERY == 0:
        q = d.qpos.numpy()                       # host copy: only for recording, not for speed
        frames.append([[round(float(a), 3), round(float(b), 3)] for a, b in q])
        if i % 100 == 0:
            spread = q[:, 0].max() - q[:, 0].min()
            print(f"t = {i * mjm.opt.timestep:.0f} s  shoulder angle spread over {NWORLD} worlds: "
                  f"{spread:.3f} rad")

q = d.qpos.numpy()
print("start spread: 0.100 rad | d.qpos.shape:", d.qpos.shape)
print("world 0 and world 255 at t = 3 s (hinge1, hinge2):", q[0], q[-1])

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"nworld": NWORLD, "dt": EVERY * float(mjm.opt.timestep), "kind": "pendulum2d",
                   "frames": frames}, f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

## 6. Why the first step is slow
Warp JIT-compiles every kernel module on first use and caches the binaries. This cell prints **your** timings (they differ on every machine) and counts the kernel launches inside one step.

Within one notebook process, only the **first** cell that steps pays the compile cost: by now Examples 1.4 to 1.5b have already compiled and loaded every kernel module, so running Example 1.6 in this process would show a first step as fast as the second. The next cell therefore writes the example to a file, and the cell after it runs it twice in **fresh** Python processes: once with the kernel cache on disk (the first step only loads the binaries), once with `--cold` (an empty, temporary cache, so every kernel module is compiled from scratch; this takes a while).

In [ ]:
%%writefile ex6_jit_timing.py
"""Example 1.6 - why the first mjw.step is slow: JIT compilation and the kernel cache.

Times the first and the second step, counts the kernel launches inside one step, and prints
where Warp keeps compiled kernels.
Run: python ex6_jit_timing.py [--cold] [--dump-json PATH]
  --cold uses an empty, temporary kernel cache, so every kernel is compiled from scratch (slow).
  --dump-json PATH writes the three step times for the page's bar chart (Rule D).
"""
import argparse
import json
import shutil
import tempfile
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

parser = argparse.ArgumentParser()
parser.add_argument("--cold", action="store_true")
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()
cold_root = None
if args.cold:
    cold_root = tempfile.mkdtemp(prefix="warp-cold-")   # Warp adds a version subfolder inside it
    wp.config.kernel_cache_dir = cold_root               # before wp.init()
wp.init()

BALL_DROP = """
<mujoco model="ball_drop">
  <option timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1" rgba=".8 .8 .8 1"/>
    <body name="ball" pos="0 0 1">
      <freejoint/>
      <geom name="ball" type="sphere" size=".1" mass="1" rgba="1 .3 .3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 8

mjm = mujoco.MjModel.from_xml_string(BALL_DROP)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=4, njmax=16)

times = []
for _ in range(3):
    t0 = time.perf_counter()
    mjw.step(m, d)
    wp.synchronize()                   # wait for the device before reading the clock
    times.append(time.perf_counter() - t0)

# Count kernel launches in one step by wrapping wp.launch / wp.launch_tiled for a moment.
counts = {"launch": 0, "launch_tiled": 0}
orig = {k: getattr(wp, k) for k in counts}


def counting(name):
    def f(*a, **kw):
        counts[name] += 1
        return orig[name](*a, **kw)
    return f


for k in counts:
    setattr(wp, k, counting(k))
mjw.step(m, d)
for k in counts:
    setattr(wp, k, orig[k])

print("device:", wp.get_device(), "| nworld:", NWORLD, "| cold cache:", args.cold)
print(f"1st step: {times[0]:8.3f} s   (compile or load every kernel module)")
print(f"2nd step: {times[1]:8.4f} s")
print(f"3rd step: {times[2]:8.4f} s")
print(f"first / second: {times[0] / times[1]:.0f}x")
print("kernel launches in one mjw.step:", counts["launch"], "wp.launch +",
      counts["launch_tiled"], "wp.launch_tiled")
print("kernel cache dir:", wp.config.kernel_cache_dir)
if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "nworld": NWORLD, "cold": args.cold,
                   "times": [round(t, 5) for t in times], "launch": counts["launch"],
                   "launch_tiled": counts["launch_tiled"]}, f)
    print("wrote", args.dump_json)
if cold_root:
    shutil.rmtree(cold_root, ignore_errors=True)   # tidy up the whole temporary cache folder

In [ ]:
# Example 1.6 - run in fresh processes: warm disk cache, then an empty cache (--cold).
!python ex6_jit_timing.py
!python ex6_jit_timing.py --cold

## 7. What is not supported, read from the code (`.deeper`)
`put_model` rejects every enum value the MJWarp enums do not contain. Diffing MuJoCo's `mjt*` enums against MJWarp's lists the unsupported features. This example imports two enums from `mujoco_warp._src.types` (private API: the name may change).

In [ ]:
# Example 1.7 - ex7_unsupported_enums.py
"""Example 1.7 - which MuJoCo features MJWarp lacks, read from the code itself.

put_model rejects any enum value that the MJWarp enums do not contain, so diffing MuJoCo's
mjt* enums against MJWarp's enums lists the unsupported features.
Run: python ex7_unsupported_enums.py [--dump-json PATH]
  --dump-json PATH writes {enum: {supported: [...], unsupported: [...]}} for the explorer.
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src import types as mjw_types   # private API: SleepPolicy, EqType are not exported

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

PAIRS = [  # (MuJoCo enum, MJWarp enum) - put_model checks each of these
    ("mjtEnableBit", mjw.EnableBit),
    ("mjtDisableBit", mjw.DisableBit),
    ("mjtDyn", mjw.DynType),
    ("mjtGain", mjw.GainType),
    ("mjtBias", mjw.BiasType),
    ("mjtTrn", mjw.TrnType),
    ("mjtSleepPolicy", mjw_types.SleepPolicy),
    ("mjtIntegrator", mjw.IntegratorType),
    ("mjtSolver", mjw.SolverType),
    ("mjtCone", mjw.ConeType),
    ("mjtEq", mjw_types.EqType),
]


def short(name):
    return name.split("_", 1)[1]           # "mjENBL_ENERGY" -> "ENERGY"


def is_sentinel(name):
    return name.startswith("mjN") or name.endswith("_UNDEFINED")   # counts and markers, not features


report = {}
for mj_name, warp_enum in PAIRS:
    mj_enum = getattr(mujoco, mj_name)
    have = {int(v) for v in warp_enum}
    members = [(n, int(v)) for n, v in mj_enum.__members__.items() if not is_sentinel(n)]
    ok = [short(n) for n, v in members if v in have]
    missing = [short(n) for n, v in members if v not in have]
    report[mj_name] = {"warp": warp_enum.__name__, "supported": ok, "unsupported": missing}
    print(f"{mj_name:15s} -> {warp_enum.__name__:15s} unsupported: {', '.join(missing) or '-'}")

n_bad = sum(len(r["unsupported"]) for r in report.values())
n_all = n_bad + sum(len(r["supported"]) for r in report.values())
print(f"{n_bad} of {n_all} enum members are rejected by put_model")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(report, f, indent=1)
    print("wrote", args.dump_json)

## 8. What an unsupported feature looks like
C MuJoCo compiles a PGS model fine; `mjw.put_model` raises `NotImplementedError`.

In [ ]:
# Example 1.8 - ex8_unsupported_pgs.py
"""Example 1.8 - what an unsupported feature looks like: the PGS solver.

C MuJoCo compiles the model fine; mjw.put_model refuses it with NotImplementedError.
Switching to the default Newton solver makes it load.
Run: python ex8_unsupported_pgs.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

XML = """
<mujoco>
  <option solver="{solver}"/>
  <worldbody>
    <geom type="plane" size="1 1 .1"/>
    <body pos="0 0 .2"><freejoint/><geom type="sphere" size=".1"/></body>
  </worldbody>
</mujoco>
"""

for solver in ["PGS", "CG", "Newton"]:
    mjm = mujoco.MjModel.from_xml_string(XML.format(solver=solver))   # C MuJoCo accepts all three
    try:
        m = mjw.put_model(mjm)
        print(f"solver={solver:6s}: put_model OK, m.opt.solver = {mjw.SolverType(m.opt.solver).name}")
    except NotImplementedError as e:
        print(f"solver={solver:6s}: {type(e).__name__}: {e}")

## Exercise 1.1: predict the landing time
Change the radius to 0.05 m and the timestep to 0.001 s. Predict when the ball first touches the floor with the free-fall formula, simulate it, and assert within 5%.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

RADIUS, TIMESTEP, H0, G = 0.05, 0.001, 1.0, 9.81
XML = f"""
<mujoco>
  <option timestep="{TIMESTEP}"/>
  <worldbody>
    <geom type="plane" size="5 5 .1"/>
    <body pos="0 0 {H0}"><freejoint/><geom type="sphere" size="{RADIUS}" mass="1"/></body>
  </worldbody>
</mujoco>
"""
t_pred = ...   # your formula
t_sim = ...    # step mjw until the first contact (d.nacon.numpy()[0] > 0), then i * TIMESTEP
if t_pred is ... or t_sim is ...:
    print("Exercise 1.1: fill in t_pred and t_sim, or open the solution")
else:
    assert abs(t_sim - t_pred) / t_pred < 0.05
    print("Exercise 1.1 passed")

In [ ]:
#@title Exercise 1.1: solution (double-click to show the code)
mjm = mujoco.MjModel.from_xml_string(XML)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=4, njmax=16)

t_pred = np.sqrt(2 * (H0 - RADIUS) / G)          # free fall until the surface touches the floor
t_sim = None
for i in range(1, 2000):
    mjw.step(m, d)
    if d.nacon.numpy()[0] > 0:                    # first step with a contact
        t_sim = i * TIMESTEP
        break
print(f"predicted {t_pred:.4f} s, simulated {t_sim:.4f} s")
assert abs(t_sim - t_pred) / t_pred < 0.05
print("Exercise 1.1 passed")

## Exercise 1.2: which models will load?
Predict `True` (loads) or `False` (rejected by `put_model`) for each of the four snippets, then run the check.

In [ ]:
import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

BASE = """
<mujoco>
  <option {opt}/>
  <worldbody>
    <geom type="plane" size="1 1 .1"/>
    <body pos="0 0 .5">
      <joint name="j" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 .3 0 0" size=".03"/>
    </body>
  </worldbody>
  {act}
</mujoco>
"""
SNIPPETS = {
    "pgs": BASE.format(opt='solver="PGS"', act=""),
    "rk4": BASE.format(opt='integrator="RK4"', act=""),
    "pid": BASE.format(opt="", act='<actuator><pid joint="j" kp="1"/></actuator>'),
    "elliptic": BASE.format(opt='cone="elliptic"', act=""),
}
predictions = {"pgs": ..., "rk4": ..., "pid": ..., "elliptic": ...}   # True = loads


def check(predictions):
    if any(v is ... for v in predictions.values()):
        print("Exercise 1.2: fill in your predictions, or open the solution")
        return
    for name, xml in SNIPPETS.items():
        mjm = mujoco.MjModel.from_xml_string(xml)          # C MuJoCo accepts all four
        try:
            mjw.put_model(mjm)
            loaded = True
        except NotImplementedError as e:
            loaded = False
            print(f"{name}: {e}")
        assert loaded == predictions[name], f"{name}: you predicted {predictions[name]}, got {loaded}"
    print("Exercise 1.2 passed")


check(predictions)

In [ ]:
#@title Exercise 1.2: solution (double-click to show the code)
predictions = {"pgs": False, "rk4": True, "pid": False, "elliptic": True}
check(predictions)

## Stretch: 65536 worlds
Copy Example 1.5, raise `NWORLD` through 1024, 4096, 16384 and 65536, warm up with one step, then time 100 steps (with `wp.synchronize()` before reading the clock) and print world-steps per second. Where does the number stop growing? Check `d.overflow` at the end.

In [ ]:
# your stretch experiment here

## Where to go next
- [Ch 2 · MuJoCo Essentials for MJWarp Users](https://github.com/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/chapters/02-mujoco-essentials.html)
- [Course map](https://github.com/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/reference/course-map.html): every Part, chapter and pipeline stage
- [Repository tour](https://github.com/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/reference/repo-tour.html): what lives where in `mujoco_warp/`
- [API at a glance](https://github.com/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/reference/api-at-a-glance.html): public names and the supported-feature explorer
- [Troubleshooting](https://github.com/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/reference/troubleshooting.html)

## Your environment summary
Print this when you ask for help or compare numbers.

In [ ]:
# Your environment, for bug reports and for comparing with the course's CPU numbers.
import importlib.metadata as md
import platform
import mujoco
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

wp.init()
print("python        :", platform.python_version())
print("mujoco-core   :", md.version("mujoco-core"), "(import mujoco ->", mujoco.__version__ + ")")
print("mujoco-warp   :", md.version("mujoco-warp"))
print("warp-lang     :", wp.__version__)
print("device        :", wp.get_device(), "| CUDA devices:", wp.get_cuda_device_count())
if wp.get_cuda_device_count() > 0:
    dev = wp.get_device("cuda:0")
    print("GPU           :", dev.name, "| memory:", round(dev.total_memory / 2**30, 1), "GiB")
    print("conditional graphs supported:", wp.is_conditional_graph_supported())
print("kernel cache  :", wp.config.kernel_cache_dir)